# Fourier-GLOF Jupyter 测试

本 notebook 用于快速测试 `GLOF_Python/glof_reconstruction.py`：

- 计算模型函数 `f(x)=x^r(-log x)^k` 的 Fourier 系数；
- 从有限 Fourier 系数进行 GLOF 重构；
- 输出误差、系数尾部，并画出重构曲线和误差曲线。

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from mpmath import mp

# 允许从仓库根目录或 GLOF_Python 目录启动 notebook
cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "GLOF_Python").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from GLOF_Python.glof_reconstruction import (
    fourier_coefficients,
    fourier_glof_reconstruct,
    linspace,
    max_abs_error,
    model_log_singularity,
    rms_error,
)

mp.dps = 80
print(f"repo_root = {repo_root}")
print(f"mp.dps = {mp.dps}")

## 1. 参数设置

In [ ]:
r = "0.5"
k = 1
alpha = "0.5"

NF = 16
theta = "0.05"
gamma = "0.12"
Q = 4 * NF

num_eval = 160
z_eval = linspace("0.005", "0.995", num_eval)

f_exact = model_log_singularity(r, k)
f_true = [f_exact(z) for z in z_eval]

print(f"NF={NF}, Q={Q}, r={r}, k={k}, alpha={alpha}, theta={theta}, gamma={gamma}")

## 2. 计算 Fourier 系数

In [ ]:
f_hat = fourier_coefficients(f_exact, NF, dps=80)

zero_mode = f_hat[NF]
zero_mode_exact = mp.gamma(k + 1) / (mp.mpf(r) + 1) ** (k + 1)
print("f_hat[0] approx =", mp.nstr(zero_mode, 16))
print("f_hat[0] exact  =", mp.nstr(zero_mode_exact, 16))
print("zero-mode err  =", mp.nstr(abs(zero_mode - zero_mode_exact), 8))

## 3. GLOF 重构

In [ ]:
result = fourier_glof_reconstruct(
    f_hat,
    r=r,
    z_eval=z_eval,
    k=k,
    alpha=alpha,
    theta=theta,
    gamma=gamma,
    Q=Q,
    dps=80,
    return_details=True,
)

R = result.values
E_inf = max_abs_error(f_true, R)
E_rms = rms_error(f_true, R)

print(result.params)
print("E_inf =", mp.nstr(E_inf, 12))
print("E_rms =", mp.nstr(E_rms, 12))

# 这个阈值用于发现明显回归；调大 NF/M 通常可以进一步降低误差。
assert E_inf < mp.mpf("0.1")

## 4. 图形诊断

In [ ]:
x_plot = [float(z) for z in z_eval]
f_plot = [float(mp.re(v)) for v in f_true]
r_plot = [float(mp.re(v)) for v in R]
err_plot = [abs(float(mp.re(a - b))) for a, b in zip(f_true, R)]

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

axes[0].plot(x_plot, f_plot, label="Exact", linewidth=2)
axes[0].plot(x_plot, r_plot, "--", label="GLOF", linewidth=2)
axes[0].set_ylabel("f(x)")
axes[0].set_title(f"Fourier-GLOF reconstruction, NF={result.params.nf}, M={result.params.m}, Q={result.params.q}")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].semilogy(x_plot, err_plot, linewidth=1.8)
axes[1].set_xlabel("x")
axes[1].set_ylabel("|f - R|")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. GLOF 系数尾部

In [ ]:
ells = list(range(len(result.coefficients)))
coeff_abs = [float(abs(c)) for c in result.coefficients]

plt.figure(figsize=(7, 4))
plt.semilogy(ells, coeff_abs, "o-", linewidth=1.8)
plt.xlabel("ell")
plt.ylabel("|c_ell|")
plt.title("GLOF coefficient magnitude")
plt.grid(True, alpha=0.3)
plt.show()

for ell, coeff in enumerate(result.coefficients[:8]):
    print(f"c[{ell}] = {mp.nstr(coeff, 14)}")

## 6. 简单收敛检查

In [ ]:
rows = []
for nf_i in [8, 12, 16]:
    f_hat_i = fourier_coefficients(f_exact, nf_i, dps=70)
    res_i = fourier_glof_reconstruct(
        f_hat_i,
        r=r,
        z_eval=z_eval,
        k=k,
        alpha=alpha,
        theta=theta,
        gamma=gamma,
        Q=4 * nf_i,
        dps=70,
        return_details=True,
    )
    err_i = max_abs_error(f_true, res_i.values)
    rows.append((nf_i, res_i.params.m, res_i.params.q, err_i))

print(" NF   M    Q    E_inf")
for nf_i, m_i, q_i, err_i in rows:
    print(f"{nf_i:3d} {m_i:3d} {q_i:4d}  {mp.nstr(err_i, 10)}")